# 05 - Baseline e previsione

I notebook precedenti hanno costruito, verificato e unito i dati. Questo
notebook chiude il percorso: calcola un risultato e lo giudica onestamente.

**Promemoria che non puoi saltare:** il notebook 04 ha dichiarato che il join
reale fra osservazioni (2024) e previsione GFS (scaricata il giorno stesso,
agosto 2026) produce zero righe, perche' NOMADS conserva solo gli ultimi
~10 giorni di run. Tutto quello che segue in questo notebook lavora quindi
sul **dataset sintetico** prodotto dal notebook 04: 18000 righe generate con
seed fisso, non un'osservazione reale. Le metriche che calcoli qui
**dimostrano il metodo** — come si confrontano baseline, come si stima un
intervallo di confidenza, come si scrive una conclusione onesta — e non
**nessuna reale capacita' di GFS sull'Italia**. Lo ripetiamo in chiusura,
perche' e' il punto piu' facile da dimenticare quando si guarda un numero.

## Passo 1 - Prerequisiti e ordine obbligatorio delle baseline

Il progetto impone un ordine preciso, non negoziabile, prima di provare
qualunque modello complesso:

**climatologia -> persistenza -> GFS grezzo -> bias correction -> (solo dopo,
eventualmente) machine learning.**

Il senso e' semplice: un modello complesso va confrontato con quello che si
ottiene quasi gratis. Se un algoritmo sofisticato non batte una media storica
per stazione, l'algoritmo non serve. Le prime due baseline (climatologia e
persistenza) non guardano nemmeno la previsione GFS: misurano quanto e'
difficile il problema di per se'. Le ultime due misurano il modello vero e
proprio, grezzo e poi corretto.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import common
import numpy as np
import pandas as pd

common.richiede("04_dataset.parquet", "01_stations.csv")
ds = pd.read_parquet(common.data_path("04_dataset.parquet"))
stazioni = pd.read_csv(common.data_path("01_stations.csv"))
train = ds[ds["split"] == "train"].copy()
test = ds[ds["split"] == "test"].copy()
print(f"train: {len(train)} righe | test: {len(test)} righe")

def metriche(previsto, osservato) -> dict:
    """Calcola MAE, RMSE e bias di una previsione contro l'osservazione."""
    e = np.asarray(previsto) - np.asarray(osservato)
    return {"mae": float(np.abs(e).mean()), "rmse": float(np.sqrt((e ** 2).mean())),
            "bias": float(e.mean()), "n": int(len(e))}

## Passo 2 - Le quattro baseline, tutte sullo stesso campione

Un confronto vale solo se e' equo: stesso campione (`test`), stesso filtro,
stessa interpolazione. Calcolarne una su un sottoinsieme diverso invenderebbe
il confronto. Le quattro baseline, nell'ordine imposto:

1. **Climatologia** — la media per stazione, calcolata SOLO sul train.
2. **Persistenza** — l'ultima osservazione disponibile nel train per quella
   stazione.
3. **GFS grezzo** — la previsione cosi' com'e', senza correzioni.
4. **Bias correction** — la correzione media per stazione, stimata SOLO sul
   train e applicata al test **senza rifare il fit**. Rifittare sul test
   sarebbe leakage, lo stesso errore dimostrato nel notebook 04.

In [ ]:
risultati = {}

# 1. Climatologia: la media per stazione calcolata SOLO sul train.
clim = train.groupby("station_id")["t2m_c_osservato"].mean()
risultati["climatologia"] = metriche(test["station_id"].map(clim), test["t2m_c_osservato"])

# 2. Persistenza: l'ultima osservazione disponibile nel train per quella stazione.
ultima = train.sort_values("valid_time_utc").groupby("station_id")["t2m_c_osservato"].last()
risultati["persistenza"] = metriche(test["station_id"].map(ultima), test["t2m_c_osservato"])

# 3. GFS grezzo, con la stessa interpolazione usata da tutto il resto.
risultati["gfs_grezzo"] = metriche(test["t2m_c_forecast"], test["t2m_c_osservato"])

# 4. Bias correction: la correzione media per stazione, stimata SOLO sul train
#    e applicata al test senza rifare il fit.
correzione = train.groupby("station_id")["errore"].mean()
corretto = test["t2m_c_forecast"] - test["station_id"].map(correzione).fillna(0.0)
risultati["bias_correction"] = metriche(corretto, test["t2m_c_osservato"])

confronto = pd.DataFrame(risultati).T[["mae", "rmse", "bias", "n"]]
print(confronto.round(3).to_string())
print("\nATTENZIONE: tutte le baseline usano lo stesso campione, lo stesso filtro")
print("e la stessa interpolazione. Un confronto fra campioni diversi non vale nulla.")

### Perche' la persistenza perde contro la climatologia

Il numero sopra puo' sembrare un bug: `persistenza` (MAE 7.80) fa peggio
di `climatologia` (MAE 5.10). Non lo e'. La persistenza qui e' l'ULTIMA
osservazione disponibile nel train, applicata come costante a ogni riga
del test: e' un valore gia' vecchio nel momento in cui inizia il periodo
di test, mentre la climatologia e' una media per stazione che rappresenta
meglio l'intero periodo. Perdere contro la climatologia e' quindi il
risultato atteso, non un errore di calcolo.

### Perche' la bias correction non ha effetto qui

Il numero sopra e' sospetto: `bias_correction` ha la STESSA MAE di
`gfs_grezzo`, alla seconda cifra decimale. Non e' un errore di calcolo: e'
la prova che, in questo campione sintetico, non c'e' quasi nulla da
correggere. Lo si vede guardando direttamente cosa la correzione ha
imparato sul train.

Il senso della bias correction non e' comunque campato in aria: nel
notebook 03 abbiamo misurato un disallineamento reale fra la quota della
stazione e la quota che il modello vede per quella cella di griglia — per
esempio AOSTA POLLEIN, 551 m reali contro 1789 m nel modello, o PIAN ROSA,
3488 m reali contro 2845 m nel modello. Quel tipo di scarto produce
esattamente un errore sistematico, specifico per stazione, ripetibile in
ogni run: e' la ragione per cui la bias correction esiste ed e' la prima
cosa da provare prima di un modello complesso. Il generatore sintetico del
notebook 04, pero', non ha riprodotto quell'effetto nei dati usati qui: ha
generato un errore vicino al rumore bianco, senza una componente sistematica
per stazione. Quello che vedi in questo notebook e' quindi il meccanismo
della correzione applicato correttamente, ma senza il caso reale che ne
giustificherebbe il beneficio.

In [ ]:
print("Correzione per stazione appresa sul train (media dell'errore GFS - osservato):")
print(correzione.round(4).to_string())
print(f"\nBias medio di GFS grezzo sul test: {risultati['gfs_grezzo']['bias']:.4f} C")
print("\nLe correzioni per stazione sono tutte vicine allo zero (ordine di 0.01-0.03 C),")
print("cosi' come il bias medio sul test. In altre parole: il generatore sintetico del")
print("notebook 04 non ha riprodotto un bias sistematico per stazione. La bias correction")
print("applica quindi una correzione quasi nulla, e il risultato e' identico a GFS grezzo")
print("PER COSTRUZIONE, non perche' il metodo abbia fallito: non c'e' errore sistematico")
print("da rimuovere in questo dato.")

## Passo 3 - Grafici: errore per lead e mappa dell'errore

Due grafici, richiesti esplicitamente dalla specifica del progetto:

- a sinistra, come cresce l'errore assoluto medio al crescere del lead time;
- a destra, **la mappa**: dove sbaglia il modello, stazione per stazione.

In [ ]:
import matplotlib.pyplot as plt
try:
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    proj = ccrs.PlateCarree()
except Exception:
    ccrs = None

fig = plt.figure(figsize=(15, 6))

# A sinistra: errore assoluto medio in funzione del lead.
ax1 = fig.add_subplot(1, 2, 1)
per_lead = test.assign(ae=test["errore"].abs()).groupby("lead_hours")["ae"].mean()
ax1.plot(per_lead.index, per_lead.values, marker="o")
ax1.set_xlabel("lead (ore)"); ax1.set_ylabel("MAE (C)")
ax1.set_title("L'errore cresce con l'orizzonte di previsione")
ax1.grid(alpha=0.3)

# A destra: dove sbaglia, sulla mappa.
mae_staz = test.assign(ae=test["errore"].abs()).groupby("station_id")["ae"].mean()
st = stazioni.set_index("station_id").loc[mae_staz.index]
ax2 = fig.add_subplot(1, 2, 2, projection=proj) if ccrs else fig.add_subplot(1, 2, 2)
kw = {"transform": proj} if ccrs else {}
if ccrs:
    margine = 0.5
    ax2.set_extent([st["lon"].min()-margine, st["lon"].max()+margine,
                    st["lat"].min()-margine, st["lat"].max()+margine], crs=proj)
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax2.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":")
sc = ax2.scatter(st["lon"], st["lat"], c=mae_staz.values, s=220,
                 cmap="YlOrRd", edgecolor="black", zorder=5, **kw)
for sid, r in st.iterrows():
    ax2.annotate(f"{r['nome']}\n{r['elev_m']:.0f} m ; MAE {mae_staz[sid]:.1f}",
                 (r["lon"], r["lat"]), fontsize=7,
                 xytext=(6, 6), textcoords="offset points", **kw)
plt.colorbar(sc, ax=ax2, label="MAE (C)", shrink=0.8)
ax2.set_title("Dove sbaglia il modello")
plt.tight_layout(); plt.show()

print("\nMAE per quota:")
print(pd.DataFrame({"mae": mae_staz, "quota_m": st["elev_m"]}).sort_values("quota_m").round(2).to_string())

## Passo 4 - Leggere il grafico

L'errore cresce col lead perche' l'incertezza iniziale si amplifica: un
piccolo errore nell'analisi di partenza si propaga e si ingigantisce ora dopo
ora. Sulla mappa, tipicamente le stazioni in quota hanno l'errore maggiore:
la cella del modello ha un'orografia mediata che non conosce la vetta, quindi
sistematicamente sovrastima o sottostima la temperatura dove il terreno vero
e' piu' estremo della sua approssimazione a griglia. **Questo errore e'
sistematico, quindi apprendibile: e' la ragione per cui il post-processing
statistico ha senso.**

## Passo 5 - Bootstrap: quanto e' solido il risultato

Un MAE calcolato su un solo campione e' un numero, non una prova. Serve un
intervallo di confidenza. Ma le righe di `test` NON sono osservazioni
indipendenti: due ore consecutive della stessa stazione si assomigliano
molto. Ricampionare per singolo record farebbe finta che ci siano molte piu'
informazioni indipendenti di quante ce ne siano davvero, e produrrebbe un
intervallo artificiosamente stretto. Per questo il ricampionamento bootstrap
qui sotto e' **per stazione-giorno**, non per record.

In [ ]:
def bootstrap_mae(previsto, osservato, gruppi, n=1000, seed=42):
    """Intervallo di confidenza del MAE, ricampionando per stazione-giorno.

    Si ricampionano i gruppi, non i singoli record: due ore consecutive
    non sono osservazioni indipendenti, e ricampionarle come tali
    restringerebbe artificiosamente l'intervallo.
    """
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"e": np.abs(np.asarray(previsto) - np.asarray(osservato)), "g": list(gruppi)})
    per_gruppo = df.groupby("g")["e"].mean()
    chiavi = per_gruppo.index.to_numpy()
    campioni = [per_gruppo[rng.choice(chiavi, len(chiavi), replace=True)].mean() for _ in range(n)]
    return float(np.percentile(campioni, 2.5)), float(np.percentile(campioni, 97.5))

gruppi = test["station_id"] + "|" + test["valid_time_utc"].dt.date.astype(str)
ic_gfs = bootstrap_mae(test["t2m_c_forecast"], test["t2m_c_osservato"], gruppi)
ic_bc = bootstrap_mae(corretto, test["t2m_c_osservato"], gruppi)

print(f"GFS grezzo      MAE {risultati['gfs_grezzo']['mae']:.2f} C  IC95% [{ic_gfs[0]:.2f}, {ic_gfs[1]:.2f}]")
print(f"Bias correction MAE {risultati['bias_correction']['mae']:.2f} C  IC95% [{ic_bc[0]:.2f}, {ic_bc[1]:.2f}]")

sovrapposti = not (ic_bc[1] < ic_gfs[0] or ic_gfs[1] < ic_bc[0])
print(f"\nGli intervalli si sovrappongono: {sovrapposti}")
if sovrapposti:
    print("=> La differenza NON e' distinguibile dal rumore su questo campione.")
    print("   Qualunque conclusione sul fatto che la correzione 'funzioni' sarebbe abusiva.")

## Passo 6 - Scrivere le metriche

Si salva tutto in `05_metrics.json`: le quattro baseline, i due intervalli
bootstrap e l'esito del confronto, con la dimensione del campione usato.

In [ ]:
import json
from datetime import datetime, timezone

uscita = {
    "baseline": risultati,
    "bootstrap": {"gfs_grezzo_ic95": list(ic_gfs), "bias_correction_ic95": list(ic_bc),
                  "intervalli_sovrapposti": bool(sovrapposti)},
    "campione": {"stazioni": int(test["station_id"].nunique()),
                 "righe_train": int(len(train)), "righe_test": int(len(test))},
    "generato_il": datetime.now(timezone.utc).isoformat(),
}
with common.data_path("05_metrics.json").open("w") as f:
    json.dump(uscita, f, indent=2, ensure_ascii=False)
print("Scritto", common.data_path("05_metrics.json"))

## Il limite di quello che hai fatto

**Quello che hai ottenuto:** il ciclo completo, dal download alla metrica,
con un metodo corretto — split temporale, fit solo sul train, confronto equo
fra quattro baseline nell'ordine imposto dal progetto, incertezza dichiarata
con un bootstrap che rispetta la dipendenza fra osservazioni vicine. E'
gia' piu' di quanto faccia la maggior parte dei confronti "veloci" che si
vedono in giro.

**Quello che NON hai ottenuto:** una prova che il post-processing funzioni.
Ricorda innanzitutto che tutte le metriche sopra sono calcolate sul
**dataset sintetico** del notebook 04: misurano se il metodo e' applicato
correttamente, non se GFS sia bravo o scarso sull'Italia. Il risultato non
conclusivo qui ha **due cause distinte**, non una sola, ed e' importante non
confonderle. La prima e' il campione minuscolo gia' visto: poche stazioni, un
solo run, pochi lead, nessuna copertura stagionale. La seconda, piu'
specifica e mostrata sopra con i numeri, e' che il dataset sintetico non
contiene un bias sistematico per stazione da correggere: la correzione
imparata sul train vale pochi centesimi di grado, il bias di GFS grezzo sul
test e' anch'esso vicino allo zero, e quindi la bias correction non ha
materialmente nulla su cui agire. In dati reali, con lo scarto di quota
misurato nel notebook 03, ci si aspetterebbe una correzione ben piu' grande
e potenzialmente utile: qui semplicemente non si presenta, perche' il
generatore sintetico non l'ha riprodotta. Se gli intervalli bootstrap si
sovrappongono — ed e' il caso di questa esecuzione — il risultato **non e'
conclusivo**, e dirlo non e' un fallimento: **e' il risultato.** Una
differenza di MAE che rientra nel rumore statistico, unita all'assenza di un
bias reale da correggere in questo campione, non autorizza a dire ne' che la
correzione del bias "funzioni" ne' che "non funzioni": non e' stata messa
alla prova in condizioni in cui potesse mostrare un effetto, per quanto la
correzione sia costruita in modo metodologicamente corretto.

**Cosa servirebbe** (dal documento `docs/04-mvp-benchmark-validation-plan.md`):
le 124 stazioni candidate invece di poche; anni interi, con train 2021-2024,
validation 2025 e test congelato; tutti i lead da 1 a 72 ore su quattro run al
giorno; segmentazione per macro-area, stagione, ora locale, fascia altimetrica
e classe di evento; e per la precipitazione un target ad accumulo, che e' un
problema piu' difficile di una temperatura istantanea perche' richiede
intervalli half-open e soglie di evento invece di un semplice errore continuo.

**Il seguito serio** e' M0 del documento `docs/08-mvp-implementation-plan.md`:
contratti dati, catalogo, idempotenza. Non e' un notebook, ed e' proprio
questa la differenza fra imparare il metodo — cosa che questo percorso ha
fatto — e costruire un sistema che quel metodo lo applica su scala, con dati
veri, per anni interi, in modo da poter finalmente rispondere alla domanda
che qui e' rimasta aperta.